# TP 1 : Pipeline de Machine Learning de Bout en Bout & Analyse de Données

Ce notebook présente la réalisation complète de notre premier TP d'MLOps (**Lab 1**).
L'objectif est de construire un pipeline d'apprentissage automatique de bout en bout en Python :

1. **Importation des librairies** nécessaires.
2. **Collecte des données** : Chargement du dataset *Breast Cancer Wisconsin*.
3. **Analyse Exploratoire des Données (EDA)** : Visualisation de la cible et analyse d'information mutuelle.
4. **Prétraitement & Feature Engineering** : Création de nouvelles variables et normalisation.
5. **Développement du Pipeline & Tuning Multi-Modèles** :
   - Définition du pipeline `Scikit-Learn`.
   - Comparaison de 3 algorithmes via `GridSearchCV` (Régression Logistique, Random Forest, SVC).
6. **Évaluation et Exportation** : Évaluation sur l'échantillon de test et sauvegarde du modèle au format `.joblib` pour notre API Flask.

### 1. Importation des Librairies

On commence par importer toutes les librairies nécessaires à la manipulation de données, la visualisation, la construction du pipeline et l'évaluation des modèles.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from joblib import dump

### 2. Chargement et Préparation des Données

#### A. Collecte des Données
On charge le jeu de données *Breast Cancer* depuis Scikit-Learn et on le structure dans un DataFrame Pandas.

In [ ]:
def get_data():
    """Charge le jeu de données Breast Cancer et retourne un DataFrame complet."""
    from sklearn.datasets import load_breast_cancer
    dataset = load_breast_cancer(as_frame=True)
    df = pd.concat([dataset['data'], dataset['target']], axis=1)
    return df

# Chargement des données
data = get_data()

# Aperçu des 5 premières lignes
data.head()

#### B. Analyse Exploratoire des Données (EDA)

Observons la répartition de notre variable cible (`target`). Dans ce dataset :
- `0` correspond à une tumeur maligne
- `1` correspond à une tumeur bénigne

In [ ]:
# Distribution de la variable cible
plt.figure(figsize=(6, 4))
sns.countplot(x=data['target'], palette='Set2')
plt.title('Répartition de la Variable Cible (0 = Maligne, 1 = Bénigne)')
plt.xlabel('Classe Target')
plt.ylabel('Effectif')
plt.show()

#### C. Importance des Variables par Information Mutuelle

L'information mutuelle (`mutual_info_classif`) permet de mesurer la quantité d'information apportée par chaque variable pour prédire le diagnostic.

In [ ]:
from sklearn.feature_selection import mutual_info_classif

# Séparation des variables explicatives (X) et de la variable cible (y)
X = data.drop("target", axis=1)
y = data["target"]

def afficher_scores_features(X, y):
    """Calcule et affiche le score d'information mutuelle de chaque variable."""
    mi_scores = mutual_info_classif(X, y, random_state=42)
    
    mi_scores_df = pd.DataFrame({
        'Variable': X.columns,
        'Information Mutuelle': mi_scores
    }).sort_values(by='Information Mutuelle', ascending=False)
    
    plt.figure(figsize=(10, 8))
    sns.barplot(x='Information Mutuelle', y='Variable', data=mi_scores_df, palette="viridis")
    plt.title('Score d'Information Mutuelle des Variables')
    plt.xlabel('Information Mutuelle')
    plt.ylabel('Variable')
    plt.show()

afficher_scores_features(X, y)

#### D. Feature Engineering (Création de Variables Personnalisées)

On crée une fonction de transformation qui génère une nouvelle variable interactive : le produit du rayon moyen par la texture moyenne.

In [ ]:
def add_combined_feature(X):
    """Génère la variable combinée 'Combined_radius_texture'."""
    X = X.copy()
    X['Combined_radius_texture'] = X['mean radius'] * X['mean texture']
    return X

### 3. Construction du Pipeline & Entraînement

#### A. Mise en Place du Pipeline et Découpage Train / Test

On enchaîne la transformation de données, la normalisation (`StandardScaler`) et le classifieur au sein d'un `Pipeline` Scikit-Learn.

In [ ]:
# Pipeline de prétraitement et scaling
preprocessing_pipeline = Pipeline([
    ('feature_engineering', FunctionTransformer(add_combined_feature)),
    ('scaler', StandardScaler())
])

# Pipeline global d'entraînement avec un modèle par défaut
training_pipeline = Pipeline(steps=[
    ('preprocessing', preprocessing_pipeline),
    ('classifier', LogisticRegression()) # Modèle temporaire
])

# Découpage en données d'entraînement (70%) et de test (30%)
X = data.drop(columns=['target'])
y = data['target']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

print(f"Données d'entraînement : {X_train.shape[0]} échantillons")
print(f"Données de test : {X_test.shape[0]} échantillons")

#### B. Comparaison Multi-Modèles avec GridSearchCV (Exercice du TP)

Comme demandé dans l'exercice, nous comparons 3 familles de modèles avec différents jeux d'hyperparamètres :
1. **Régression Logistique** (`LogisticRegression`) : `C` $\in [0.1, 1.0, 10.0]$
2. **Forêt Aléatoire** (`RandomForestClassifier`) : `n_estimators` $\in [50, 100, 200]$, `max_depth` $\in [\text{None}, 10, 20]$
3. **Support Vector Machine** (`SVC`) : `C` $\in [0.1, 1.0, 10.0]$, `kernel` $\in ['linear', 'rbf']$

In [ ]:
# Grille d'hyperparamètres combinant les 3 algorithmes
multi_models_grid = [
    {
        'classifier': [LogisticRegression(max_iter=1000, random_state=42)],
        'classifier__C': [0.1, 1.0, 10.0]
    },
    {
        'classifier': [RandomForestClassifier(random_state=42)],
        'classifier__n_estimators': [50, 100, 200],
        'classifier__max_depth': [None, 10, 20]
    },
    {
        'classifier': [SVC(random_state=42)],
        'classifier__C': [0.1, 1.0, 10.0],
        'classifier__kernel': ['linear', 'rbf']
    }
]

# Recherche des meilleurs hyperparamètres par validation croisée (5 folds)
grid_search_multi = GridSearchCV(training_pipeline, multi_models_grid, cv=5, n_jobs=1, verbose=1)
grid_search_multi.fit(X_train, y_train)

print(f"\nMeilleurs hyperparamètres sélectionnés : {grid_search_multi.best_params_}")
print(f"Meilleur score de validation croisée (Accuracy) : {grid_search_multi.best_score_:.4f}")

best_model = grid_search_multi.best_estimator_

#### C. Évaluation Finale du Meilleur Modèle sur le Jeu de Test

Testons à présent les performances de généralisation du modèle sélectionné sur le jeu de test inédit.

In [ ]:
def evaluate_model(model, X_test, y_test):
    """Calcule et affiche les métriques de performance sur le jeu de test."""
    y_pred = model.predict(X_test)
    
    print(f"Accuracy sur le jeu de test : {accuracy_score(y_test, y_pred):.4f}\n")
    print("--- Rapport de Classification ---")
    print(classification_report(y_test, y_pred))
    
    # Matrice de confusion
    conf_matrix = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(7, 5))
    sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Maligne (0)', 'Bénigne (1)'],
                yticklabels=['Maligne (0)', 'Bénigne (1)'])
    plt.title('Matrice de Confusion (Jeu de Test)')
    plt.xlabel('Prédiction')
    plt.ylabel('Valeur Réelle')
    plt.show()

# Lancement de l'évaluation
evaluate_model(best_model, X_test, y_test)

#### D. Sauvegarde du Pipeline pour Déploiement API

Pour terminer, on sauvegarde l'ensemble du pipeline (préparations + modèle) dans un fichier `.joblib` afin de l'exploiter dans notre API d'inférence Flask.

In [ ]:
# Sauvegarde de l'artefact de modèle
dump(best_model, 'best_cancer_model_pipeline.joblib')
print("Artefact 'best_cancer_model_pipeline.joblib' sauvegardé avec succès !")